# Woche 06 – Vorlesungsdemo: Läuft diese Person gegen die Wand? 🏃‍♀️🧱

**Methods of Data Science · Wintersemester 2026/27**

Beim Marathon gibt es einen gefürchteten Moment: Irgendwann sind die Kohlenhydratspeicher leer, die Beine werden schwer und das Tempo bricht ein. Läufer:innen nennen das **„hit the wall“** – gegen die Wand laufen.

Stellt euch vor, ihr steht beim **Boston Marathon** genau bei der **Hälfte (km 21,1)**. Für jede Person, die vorbeiläuft, kennt ihr:

| Was wir bei km 21,1 wissen | Variable |
|---|---|
| Alter | `age` |
| Geschlecht | `gender` (F / M) |
| Zeit für die erste Hälfte in Minuten | `half_time` |
| Um wie viel Prozent war die Person zwischen km 10 und km 21,1 langsamer als auf den ersten 10 km? | `pace_change` |

**Frage:** Läuft diese Person in der zweiten Hälfte gegen die Wand?
Wir sagen: Jemand läuft gegen die Wand (`hit_the_wall = 1`), wenn die **zweite Hälfte mindestens 20 % länger dauert als die erste**.

**Daten:** Ergebnisse des Boston Marathon 2014 (Boston Athletic Association), gesammelt von B. Mill: https://github.com/llimllib/bostonmarathon. Der Datensatz ist bereits aufbereitet und enthält nur Läufer:innen, die das Ziel erreicht haben.

## Vorbereitung

*Führt den Code unten aus.*

In [ ]:
# Imports
import numpy as np
import random
# Seed
np.random.seed(42)  # Set seed for NumPy
random.seed(42) # Set seed for random module

import pandas as pd
import matplotlib.pyplot as plt

## Daten laden

Zuerst laden wir den Marathon-Datensatz.

*Führt den Code unten aus.*

In [ ]:
#Loading the data from a csv file
data = pd.read_csv("https://raw.githubusercontent.com/kbrennig/MODS_WS26_27/refs/heads/main/data/BostonMarathon2014.csv")
data.head()

## Daten ansehen

*Führt den Code unten aus.*

In [ ]:
print(data.describe())
print(data.dtypes)

In [ ]:
# Wie viele Läufer:innen laufen "gegen die Wand"?
print(data["hit_the_wall"].value_counts(normalize=True))

## Trainings- und Testdaten erstellen

Wir teilen die Daten in 80 % Trainingsdaten und 20 % Testdaten auf.

*Führt den Code unten aus.*

In [ ]:
from sklearn.model_selection import train_test_split

data_training, data_test = train_test_split(data, test_size=0.2, random_state=42)

## Daten transformieren

Machine-Learning-Modelle können nicht mit Text-Kategorien arbeiten, sie brauchen Zahlen. Deshalb wandeln wir die kategoriale Variable `gender` per Dummy-Encoding um. Die erste Kategorie (F) wird weggelassen, es bleibt die Spalte `gender_M` (1 = Mann, 0 = Frau).

Der Encoder lernt die Kategorien nur auf den Trainingsdaten (`fit_transform`) und wendet sie dann auf die Testdaten an (`transform`).

*Führt den Code unten aus.*

In [ ]:
from sklearn.preprocessing import OneHotEncoder
# Dummy encoding / One-hot encoding

categorical_columns = [
    "gender"
]

# Creating the OneHotEncoder object (returns a pandas DataFrame)
encoder = OneHotEncoder(drop='first', handle_unknown='infrequent_if_exist', sparse_output=False).set_output(transform="pandas")

# Training data: learn the categories, encode them and replace the original columns
data_training = data_training.drop(columns=categorical_columns).join(encoder.fit_transform(data_training[categorical_columns]))

# Test data: apply the same encoding
data_test = data_test.drop(columns=categorical_columns).join(encoder.transform(data_test[categorical_columns]))

# define target variable
X_train = data_training.drop(columns="hit_the_wall")
y_train = data_training["hit_the_wall"]
X_test = data_test.drop(columns="hit_the_wall")
y_test = data_test["hit_the_wall"]

X_train.head()

# Demo 1: Einen Baum wachsen lassen und lesen

Wir erzeugen mit `DecisionTreeClassifier` einen Klassifikationsbaum (CART) und trainieren ihn mit `fit` auf den Trainingsdaten. Damit wir ihn gut lesen können, begrenzen wir die Tiefe des Baums etwas.

*Führt den Code unten aus.*

In [ ]:
from sklearn.tree import DecisionTreeClassifier
model_01 = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)

In [ ]:
from sklearn.tree import plot_tree
plt.figure(figsize=(24,9))
plot_tree(model_01, feature_names=X_train.columns, impurity=False, filled=True, rounded=True, proportion=True, class_names=["not hit the wall", "hit the wall"], fontsize=11)
plt.show()

**So lest ihr einen Knoten**

* **Erste Zeile:** die Split-Bedingung, z. B. `pace_change <= 4.515`. Ist sie **wahr, geht es nach links**, sonst nach rechts.
* **samples:** Anteil der Trainingsdaten in diesem Knoten.
* **value:** Anteile der Klassen `[not hit the wall, hit the wall]` in diesem Knoten.
* **class:** die Mehrheitsklasse = die Vorhersage für alle, die hier landen.
* **Farbe:** orange = not hit the wall, blau = hit the wall. Je kräftiger, desto reiner der Knoten.

👉 *Was lernt ihr über Läufer:innen, die schon in der ersten Hälfte langsamer werden?*

### Vorhersage für einen einzelnen Läufer

Verschiebt die Regler und beobachtet die Vorhersage. Die Wahrscheinlichkeit ist einfach der Anteil der Läufer:innen mit „hit the wall“ im Blatt, in dem die Person landet. Das Alter taucht im Baum nicht auf, deshalb gibt es dafür keinen Regler.

*Führt die Zelle aus – den Code dahinter müsst ihr nicht kennen.*

In [ ]:
#@title 🎚 Regler: Vorhersage für einen Läufer { display-mode: "form" }
from ipywidgets import interact, IntSlider, FloatSlider, Dropdown

def vorhersage_laeufer(geschlecht="F", half_time=105, pace_change=2.0):
    runner = pd.DataFrame({"age": [40], "half_time": [half_time], "pace_change": [pace_change],
                           "gender_M": [1.0 if geschlecht == "M" else 0.0]})[X_train.columns]
    p_wall = model_01.predict_proba(runner)[0, 1]
    print("Vorhersage:", "hit the wall 🧱" if p_wall > 0.5 else "not hit the wall 🏃")
    print(f"Wahrscheinlichkeit für 'hit the wall': {p_wall:.0%}  (Anteil der Läufer:innen mit 'hit the wall' im Blatt)")

interact(vorhersage_laeufer,
         geschlecht=Dropdown(options=["F", "M"], value="F", description="gender"),
         half_time=IntSlider(value=105, min=65, max=200, step=5, description="half_time"),
         pace_change=FloatSlider(value=2.0, min=-10, max=20, step=0.5, description="pace_change"));

# Demo 2: Overfitting

Was passiert, wenn wir das Wachstum des Baums **nicht** begrenzen? Wir trainieren einen Baum ohne Hyperparameter (`model_full`) und vergleichen die Accuracy auf Trainings- und Testdaten.

*Führt den Code unten aus.*

In [ ]:
from sklearn.metrics import accuracy_score

model_full = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)

print("Blätter:", model_full.get_n_leaves())
print("Accuracy Training:", round(accuracy_score(y_train, model_full.predict(X_train)), 3))
print("Accuracy Test:    ", round(accuracy_score(y_test, model_full.predict(X_test)), 3))

👉 *Der Baum ohne Grenzen lernt die Trainingsdaten fast perfekt auswendig – auf den Testdaten ist er deutlich schlechter. Das ist Overfitting.*

## Lösung 1: Stopping Rules

Mit Stopping Rules begrenzen wir das Wachstum des Baums schon **vor** dem Training:

* `max_depth`: maximale Tiefe des Baums (die Wurzel hat Tiefe 0)
* `min_samples_split`: Mindestanzahl an Beobachtungen in einem Knoten, damit ein Split versucht wird
* `min_samples_leaf`: Mindestanzahl an Beobachtungen in jedem Blatt

*Führt die Zelle aus und verschiebt die Regler. Beobachtet die Accuracy auf Trainings- und Testdaten.*

In [ ]:
#@title 🎚 Regler: Stopping Rules { display-mode: "form" }
from ipywidgets import interact, IntSlider

def zeige_baum(m):
    print(f"Blätter: {m.get_n_leaves()}  |  Accuracy Training: {accuracy_score(y_train, m.predict(X_train)):.1%}"
          f"  |  Accuracy Test: {accuracy_score(y_test, m.predict(X_test)):.1%}")
    if m.get_n_leaves() <= 16:
        plt.figure(figsize=(22, 8))
        plot_tree(m, feature_names=X_train.columns, impurity=False, filled=True, rounded=True,
                  proportion=True, class_names=["not hit the wall", "hit the wall"], fontsize=10)
        plt.show()
    else:
        print("(Der Baum ist zu groß zum Zeichnen.)")

def stopping_rules(max_depth=3, min_samples_split=2, min_samples_leaf=1):
    m = DecisionTreeClassifier(max_depth=max_depth, min_samples_split=min_samples_split,
                               min_samples_leaf=min_samples_leaf, random_state=42).fit(X_train, y_train)
    zeige_baum(m)

interact(stopping_rules,
         max_depth=IntSlider(value=3, min=1, max=20),
         min_samples_split=IntSlider(value=2, min=2, max=1002, step=50),
         min_samples_leaf=IntSlider(value=1, min=1, max=501, step=25));

## Lösung 2: Pruning

Beim Pruning lassen wir den Baum zuerst voll wachsen und schneiden ihn dann zurück. In scikit-learn steuert das der Hyperparameter `ccp_alpha`:

* `ccp_alpha = 0`: kein Pruning – der volle Baum bleibt stehen
* je größer `ccp_alpha`, desto mehr Äste werden abgeschnitten und desto kleiner wird der Baum

*Führt die Zelle aus und schiebt den Regler langsam nach rechts.*

In [ ]:
#@title 🎚 Regler: Pruning { display-mode: "form" }
from ipywidgets import interact, FloatSlider

def pruning(ccp_alpha=0.0):
    m = DecisionTreeClassifier(ccp_alpha=ccp_alpha, random_state=42).fit(X_train, y_train)
    zeige_baum(m)

interact(pruning, ccp_alpha=FloatSlider(value=0.0, min=0.0, max=0.005, step=0.0001, readout_format=".4f"));

# Demo 3: Accuracy und Confusion Matrix

Für die Evaluation trainieren wir einen Baum mit Stopping Rules (`model_02`).

*Führt den Code unten aus.*

In [ ]:
model_02 = DecisionTreeClassifier(min_samples_split=60, min_samples_leaf=20, max_depth=5, random_state=42).fit(X_train, y_train)

#### Vorhersagen für die Testdaten

Wir lassen uns zuerst mit `predict_proba` die Wahrscheinlichkeiten für die Testdaten ausgeben und wandeln sie dann mit dem Schwellenwert (Threshold) 0,5 in Klassen um.

* **`predictions_test_02`** enthält für jede Läuferin und jeden Läufer im Testset **zwei Wahrscheinlichkeiten**: Spalte 0 = Wahrscheinlichkeit für „not hit the wall“ (Klasse 0), Spalte 1 = Wahrscheinlichkeit für „hit the wall“ (Klasse 1). Beide zusammen ergeben 1.
* **`predictions_test_02_binary`** enthält die **vorhergesagte Klasse** (0 oder 1): Ist die Wahrscheinlichkeit für „hit the wall“ (`predictions_test_02[:,1]`) größer als 0,5, wird daraus eine 1, sonst eine 0.

Für **Accuracy, Confusion Matrix, Precision und Recall** verwenden wir die Klassen `predictions_test_02_binary` (Threshold 0,5). Für die **ROC-Kurve und die AUC** in Demo 4 verwenden wir die Wahrscheinlichkeiten `predictions_test_02[:,1]`.

*Führt den Code unten aus.*

In [ ]:
predictions_test_02 = model_02.predict_proba(X_test)
predictions_test_02_binary = np.where(predictions_test_02[:,1] > 0.5, 1, 0)

In [ ]:
# Die ersten fünf Läufer:innen im Testset
print("predictions_test_02 (Spalte 0: not hit the wall, Spalte 1: hit the wall):")
print(predictions_test_02[:5].round(3))
print("predictions_test_02_binary:", predictions_test_02_binary[:5])
print("wahre Klasse (y_test):     ", y_test[:5].values)

#### Accuracy und Confusion Matrix

Die **Accuracy** gibt an, welcher Anteil unserer Vorhersagen auf den Testdaten richtig ist. Dafür vergleichen wir die vorhergesagten Klassen `predictions_test_02_binary` mit den wahren Klassen `y_test`.

Die **Confusion Matrix** zeigt genauer, welche Fehler wir machen: Zeilen = wahre Klasse, Spalten = vorhergesagte Klasse, jeweils 0 = „not hit the wall“ und 1 = „hit the wall“.

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import ConfusionMatrixDisplay

# Calculate the accuracy of the model
accuracy_model_02 = accuracy_score(y_test, predictions_test_02_binary)
print(accuracy_model_02)

# Display the confusion matrix
ConfusionMatrixDisplay.from_predictions(y_test, predictions_test_02_binary)

#### Baseline

Wie gut wäre ein Modell, das einfach **immer** die häufigste Klasse („not hit the wall“) vorhersagt?

In [ ]:
baseline = y_test.value_counts(normalize=True).max()
print(baseline)

#### Warum reicht die Accuracy nicht?

Unser Modell liegt nur wenige Prozentpunkte über der Baseline. Die Baseline erreicht man ohne jedes Modell, indem man einfach immer „not hit the wall“ vorhersagt – weil die meisten Läufer:innen nicht gegen die Wand laufen.

Bei solchen **unbalancierten Klassen** sieht die Accuracy gut aus, auch wenn das Modell die seltenere Klasse kaum erkennt. In der Confusion Matrix seht ihr das unten links: Viele Läufer:innen, die tatsächlich gegen die Wand gelaufen sind (wahre Klasse 1), hat das Modell als „not hit the wall“ (0) vorhergesagt. Davon erzählt die Accuracy nichts. Deshalb schauen wir uns zusätzlich **Precision** und **Recall** an.

#### Precision und Recall

Beide Kennzahlen schauen auf die positive Klasse 1 = „hit the wall“ und werden – wie die Accuracy – mit den Klassen `predictions_test_02_binary` (Threshold 0,5) berechnet:

* **Precision** = TP / (TP + FP): Von allen Läufer:innen, für die das Modell „hit the wall“ vorhersagt, wie viele laufen tatsächlich gegen die Wand?
* **Recall** = TP / (TP + FN): Von allen Läufer:innen, die tatsächlich gegen die Wand laufen, wie viele erkennt das Modell?

*Führt den Code unten aus.*

In [ ]:
from sklearn.metrics import precision_score, recall_score

precision_model_02 = precision_score(y_test, predictions_test_02_binary)
recall_model_02 = recall_score(y_test, predictions_test_02_binary)

print("Precision:", round(precision_model_02, 3))
print(f"  -> Von allen, für die wir 'hit the wall' vorhersagen, laufen {precision_model_02:.0%} tatsächlich gegen die Wand.")
print("Recall:   ", round(recall_model_02, 3))
print(f"  -> Von allen, die tatsächlich gegen die Wand laufen, erkennt das Modell {recall_model_02:.0%}.")

👉 *Was bedeutet das?* Wenn das Modell „hit the wall“ sagt, liegt es meistens richtig (Precision ca. 0,69). Es **übersieht** aber mehr als die Hälfte der Läufer:innen, die tatsächlich gegen die Wand laufen (Recall ca. 0,43). Beide Werte hängen vom Threshold 0,5 ab – was passiert, wenn wir ihn verschieben, sehen wir in Demo 4.

# Demo 4: Threshold, ROC und AUC

Bisher haben wir mit dem Threshold 0,5 gearbeitet. Jetzt verschieben wir ihn: Der Regler nimmt die Wahrscheinlichkeiten `predictions_test_02[:,1]` und erzeugt für jeden Threshold neue Klassen (0 / 1). Jeder Threshold ergibt eine eigene Confusion Matrix – und damit einen Punkt auf der ROC-Kurve.

Verschiebt den Regler und beobachtet, wie sich die Confusion Matrix und der rote Punkt auf der ROC-Kurve verändern.

*Führt die Zelle aus – den Code dahinter müsst ihr nicht kennen.*

In [ ]:
#@title 🎚 Regler: Threshold und ROC-Kurve { display-mode: "form" }
from ipywidgets import interact, FloatSlider
from sklearn.metrics import roc_curve, confusion_matrix

fpr, tpr, _ = roc_curve(y_test, predictions_test_02[:,1])

def threshold_zeigen(threshold=0.5):
    pred = np.where(predictions_test_02[:,1] > threshold, 1, 0)
    tn, fp, fn, tp = confusion_matrix(y_test, pred).ravel()
    fig, ax = plt.subplots(1, 2, figsize=(13, 5))
    ConfusionMatrixDisplay.from_predictions(y_test, pred, ax=ax[0], colorbar=False)
    ax[0].set_title(f"Threshold = {threshold:.2f}  |  Accuracy = {(tp + tn) / len(pred):.1%}")
    ax[1].plot(fpr, tpr, label="ROC-Kurve")
    ax[1].plot([0, 1], [0, 1], "--", color="grey", label="Raten")
    ax[1].scatter([fp / (fp + tn)], [tp / (tp + fn)], s=120, color="red", zorder=3, label="aktueller Threshold")
    ax[1].set_xlabel("False Positive Rate"); ax[1].set_ylabel("True Positive Rate (Recall)")
    ax[1].legend(loc="lower right"); ax[1].grid(alpha=0.3)
    plt.show()

interact(threshold_zeigen, threshold=FloatSlider(value=0.5, min=0.0, max=0.95, step=0.05));

#### ROC-Kurve und AUC

Für die ROC-Kurve und die AUC verwenden wir die **Wahrscheinlichkeiten** `predictions_test_02[:,1]` (Spalte 1 = Wahrscheinlichkeit für „hit the wall“) – **nicht** die Klassen `predictions_test_02_binary`. Der Grund: Die ROC-Kurve betrachtet alle möglichen Thresholds. In `predictions_test_02_binary` steckt aber nur noch das Ergebnis für den Threshold 0,5.

*Führt den Code unten aus.*

In [ ]:
from sklearn.metrics import roc_auc_score
from sklearn.metrics import RocCurveDisplay

# Calculate and Print the AUC score
auc_score = roc_auc_score(y_test, predictions_test_02[:,1])
print("AUC Score:", auc_score)

#plot ROC curve
RocCurveDisplay.from_predictions(y_test, predictions_test_02[:,1], plot_chance_level=True)

## Zusammenfassung

1. Ein Entscheidungsbaum teilt die Daten Schritt für Schritt in immer reinere Gruppen auf.
2. Ein Baum ohne Grenzen **overfittet**: Auf den Trainingsdaten ist er fast perfekt, auf neuen Daten schlecht. Stopping Rules und Pruning helfen dagegen.
3. Vergleicht die Accuracy immer mit der **Baseline**. Bei unbalancierten Klassen ist sie allein irreführend.
4. Confusion Matrix, Precision, Recall und ROC/AUC zeigen, **welche** Fehler das Modell macht.

➡️ In der Übung wendet ihr all das auf den **Micro-Credit**-Datensatz an.